In [ ]:
import requests
import pandas as pd

BASE = "https://data360api.worldbank.org"

def search_indicators(topic, keyword="*", top=20):
    """Buscar indicadores por tema."""
    resp = requests.post(f"{BASE}/data360/searchv2", json={
        "count": False,
        "filter": f"series_description/topics/any(t: t/name eq '{topic}') and type eq 'indicator'",
        "select": "series_description/idno, series_description/name, series_description/database_id",
        "search": keyword,
        "top": top
    })
    return resp.json()

def get_data(database_id, indicator, countries=None, time_from=None, time_to=None):
    """Obtener datos con paginacion automatica."""
    params = {"DATABASE_ID": database_id, "INDICATOR": indicator}
    if countries:
        params["REF_AREA"] = countries
    if time_from:
        params["timePeriodFrom"] = time_from
    if time_to:
        params["timePeriodTo"] = time_to

    all_data = []
    skip = 0
    while True:
        params["skip"] = skip
        resp = requests.get(f"{BASE}/data360/data", params=params)
        values = resp.json().get("value", [])
        all_data.extend(values)
        if len(values) < 1000:
            break
        skip += 1000

    return pd.DataFrame(all_data)

In [12]:
search_indicators('poverty')

{'@odata.context': "https://itsda-dataexp-prd.search.windows.net/indexes('data360-metadata-vector')/$metadata#docs(*)",
 '@odata.count': 1501,
 'value': [{'@search.score': 43.18896,
   'series_description': {'idno': 'WB_SSGD_POVERTY_RATIO_NPL',
    'name': 'Poverty headcount ratio at national poverty lines',
    'database_id': 'WB_SSGD'}},
  {'@search.score': 42.92316,
   'series_description': {'idno': 'WB_WDI_SI_POV_GAPS',
    'name': 'Poverty gap at $3.00 a day (2021 PPP) (%)',
    'database_id': 'WB_WDI'}},
  {'@search.score': 42.03869,
   'series_description': {'idno': 'WB_WDI_SI_POV_NAHC',
    'name': 'Poverty headcount ratio at national poverty lines (% of population)',
    'database_id': 'WB_WDI'}},
  {'@search.score': 41.24271,
   'series_description': {'idno': 'WB_WDI_SI_POV_LMIC',
    'name': 'Poverty headcount ratio at $4.20 a day (2021 PPP) (% of population)',
    'database_id': 'WB_WDI'}},
  {'@search.score': 40.79392,
   'series_description': {'idno': 'WB_WDI_SI_POV_UMIC'